# AMM Simulator — Multi-Pool Competition

Simulates realistic DeFi market structure: multiple AMM pools competing for the same retail order flow.
Retail takers route orders optimally via KKT bisection — maximising their output across all active pools.

**Pool setup: 2 CP AMM + 2 Linear AMM (4 pools total)**

Experiments:
1. CP AMMs with different fees competing
2. Linear AMMs with different spreads competing  
3. Mixed CP + Linear competition
4. Comparison vs single-pool baseline

## 0. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

keys_to_remove = [k for k in sys.modules if 'amm_sim' in k]
for k in keys_to_remove:
    del sys.modules[k]

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from amm_sim.types import SimParams
from amm_sim.env import make_env

from amm_sim.amms.constant_product import (
    CONSTANT_PRODUCT_AMM, CPParams,
    cp_arb_solver, cp_edge,
    cp_marginal_inverse_buy, cp_marginal_inverse_sell,
)
from amm_sim.amms.linear import (
    LINEAR_AMM, LinearParams,
    linear_arb_solver, linear_edge,
    linear_marginal_inverse_buy, linear_marginal_inverse_sell,
)

print('JAX version :', jax.__version__)
print('JAX devices :', jax.devices())

## 1. Shared Setup

In [ ]:
sim_params = SimParams(
    sigma=0.001,
    num_steps=1_000,
    max_orders=16,
    lam=0.8,
    mu=1.0,
    sigma_ln=0.5,
    phi=0.0,
)

N_EPISODES = 200

def run_batch(env, n=N_EPISODES, seed=42):
    """Run N episodes, return per-episode edge sums."""
    keys = jax.random.split(jax.random.PRNGKey(seed), n)
    _, trajs = env.batch_rollout(keys, sim_params)
    jax.block_until_ready(trajs)
    return {
        'total'  : np.array(trajs.total_edge).sum(axis=1),
        'arb'    : np.array(trajs.arb_edge).sum(axis=1),
        'retail' : np.array(trajs.retail_edge).sum(axis=1),
    }

def print_results(name, r):
    print(f"{name}")
    print(f"  total  : {r['total'].mean():8.2f} ± {r['total'].std():.2f}")
    print(f"  arb    : {r['arb'].mean():8.2f}")
    print(f"  retail : {r['retail'].mean():8.2f}")
    print(f"  arb<=0 : {bool((np.array(r['arb']) <= 1e-4).all())}")

print('Setup done.')

---
## Experiment 1: CP AMMs with Different Fees

Real-world analogy: multiple Uniswap V2 pools for the same token pair but different fee tiers.
Retail takers split orders across pools to get the best combined execution price.

**4 CP AMMs**: 10 bps / 20 bps / 30 bps / 50 bps

In [ ]:
cp_10bps = CPParams(fee_plus=0.001, fee_minus=0.001, init_x=100.0, init_y=10_000.0)
cp_20bps = CPParams(fee_plus=0.002, fee_minus=0.002, init_x=100.0, init_y=10_000.0)
cp_30bps = CPParams(fee_plus=0.003, fee_minus=0.003, init_x=100.0, init_y=10_000.0)
cp_50bps = CPParams(fee_plus=0.005, fee_minus=0.005, init_x=100.0, init_y=10_000.0)

cp_inv = (cp_marginal_inverse_buy, cp_marginal_inverse_sell)

env_4cp = make_env(
    amm_specs   = [CONSTANT_PRODUCT_AMM] * 4,
    amm_params  = [cp_10bps, cp_20bps, cp_30bps, cp_50bps],
    arb_solvers = [cp_arb_solver] * 4,
    edge_fns    = [cp_edge] * 4,
    marginal_inverse_fns = [cp_inv] * 4,
)

# Warm up JIT
keys_warm = jax.random.split(jax.random.PRNGKey(0), 4)
_ = env_4cp.batch_rollout(keys_warm, sim_params)
jax.block_until_ready(_)

r_4cp = run_batch(env_4cp)
print_results('4 CP AMMs (10/20/30/50 bps)', r_4cp)

---
## Experiment 2: Linear AMMs with Different Spreads

Real-world analogy: multiple market makers posting different bid-ask spreads.
Tighter spread pool attracts more retail flow but also more arbitrage.

**4 Linear AMMs**: Z± = 0.1 / 0.2 / 0.3 / 0.5

In [ ]:
def lin_params(z):
    return LinearParams(
        lam_pp=0.01, lam_mm=0.01,
        lam_pm=0.005, lam_mp=0.005,
        init_z_plus=z, init_z_minus=z,
    )

lin_inv = (linear_marginal_inverse_buy, linear_marginal_inverse_sell)

env_4lin = make_env(
    amm_specs   = [LINEAR_AMM] * 4,
    amm_params  = [lin_params(0.1), lin_params(0.2),
                   lin_params(0.3), lin_params(0.5)],
    arb_solvers = [linear_arb_solver] * 4,
    edge_fns    = [linear_edge] * 4,
    marginal_inverse_fns = [lin_inv] * 4,
)

_ = env_4lin.batch_rollout(keys_warm, sim_params)
jax.block_until_ready(_)

r_4lin = run_batch(env_4lin)
print_results('4 Linear AMMs (Z=0.1/0.2/0.3/0.5)', r_4lin)

---
## Experiment 3: Mixed Competition — 2 CP + 2 Linear

Real-world analogy: a token pair traded on both Uniswap (CP AMM) and
a professional market maker (Linear AMM) simultaneously.
Takers route optimally across fundamentally different pool types.

**Pool 0**: CP AMM 20 bps  
**Pool 1**: CP AMM 30 bps  
**Pool 2**: Linear AMM Z± = 0.2  
**Pool 3**: Linear AMM Z± = 0.3

In [ ]:
env_mixed = make_env(
    amm_specs   = [CONSTANT_PRODUCT_AMM, CONSTANT_PRODUCT_AMM,
                   LINEAR_AMM,           LINEAR_AMM],
    amm_params  = [cp_20bps, cp_30bps,
                   lin_params(0.2), lin_params(0.3)],
    arb_solvers = [cp_arb_solver,     cp_arb_solver,
                   linear_arb_solver, linear_arb_solver],
    edge_fns    = [cp_edge,     cp_edge,
                   linear_edge, linear_edge],
    marginal_inverse_fns = [cp_inv,  cp_inv,
                             lin_inv, lin_inv],
)

_ = env_mixed.batch_rollout(keys_warm, sim_params)
jax.block_until_ready(_)

r_mixed = run_batch(env_mixed)
print_results('2 CP + 2 Linear (mixed)', r_mixed)

---
## Experiment 4: Baseline Comparison — Single Pool

Compare multi-pool results vs a single pool with no competition.
In a single-pool setting, the LP captures all retail flow but also absorbs all arb.

In [ ]:
# Single CP AMM 30 bps — no competition
# Use two identical pools as a proxy (engine requires at least 2)
env_single_cp = make_env(
    amm_specs   = [CONSTANT_PRODUCT_AMM, CONSTANT_PRODUCT_AMM],
    amm_params  = [cp_30bps, cp_30bps],
    arb_solvers = [cp_arb_solver] * 2,
    edge_fns    = [cp_edge] * 2,
    marginal_inverse_fns = [cp_inv] * 2,
)

env_single_lin = make_env(
    amm_specs   = [LINEAR_AMM, LINEAR_AMM],
    amm_params  = [lin_params(0.3), lin_params(0.3)],
    arb_solvers = [linear_arb_solver] * 2,
    edge_fns    = [linear_edge] * 2,
    marginal_inverse_fns = [lin_inv] * 2,
)

r_single_cp  = run_batch(env_single_cp)
r_single_lin = run_batch(env_single_lin)

print_results('Single CP AMM 30bps (baseline)',   r_single_cp)
print()
print_results('Single Linear AMM Z=0.3 (baseline)', r_single_lin)

---
## Summary Plot

In [ ]:
results = {
    'Single CP\n30bps':        r_single_cp,
    'Single Linear\nZ=0.3':   r_single_lin,
    '4 CP AMMs\n10-50bps':    r_4cp,
    '4 Linear AMMs\nZ=0.1-0.5': r_4lin,
    '2 CP + 2 Linear\nmixed': r_mixed,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Multi-Pool Competition: Total Edge per Episode', fontsize=13)

# Left: distribution
ax = axes[0]
colors = ['steelblue', 'seagreen', 'tomato', 'gold', 'mediumpurple']
for (name, r), color in zip(results.items(), colors):
    ax.hist(r['total'], bins=25, alpha=0.5, color=color,
            label=f"{name.replace(chr(10), ' ')}  μ={r['total'].mean():.1f}",
            edgecolor='white')
    ax.axvline(r['total'].mean(), color=color, lw=1.5, linestyle='--')
ax.set_xlabel('Total Edge per Episode')
ax.set_ylabel('Count')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: arb vs retail breakdown
ax = axes[1]
names  = [n.replace('\n', '\n') for n in results.keys()]
arbs   = [r['arb'].mean()    for r in results.values()]
rets   = [r['retail'].mean() for r in results.values()]
x = np.arange(len(names))
w = 0.35
ax.bar(x - w/2, arbs, w, label='Arb edge',    color='tomato',   alpha=0.8)
ax.bar(x + w/2, rets, w, label='Retail edge', color='seagreen', alpha=0.8)
ax.axhline(0, color='black', lw=0.8, linestyle=':')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=8)
ax.set_ylabel('Mean Edge per Episode')
ax.set_title('Arb vs Retail Breakdown')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Setup':<30} {'Total':>8}  {'Arb':>8}  {'Retail':>8}  {'Arb/Retail':>10}")
print('-' * 72)
for name, r in results.items():
    ratio = abs(r['arb'].mean()) / r['retail'].mean() if r['retail'].mean() > 0 else float('nan')
    print(f"{name.replace(chr(10), ' '):<30} "
          f"{r['total'].mean():>8.2f}  "
          f"{r['arb'].mean():>8.2f}  "
          f"{r['retail'].mean():>8.2f}  "
          f"{ratio:>10.3f}")

---
## Asymmetric Fee Experiment: CP AMM Directional Pricing

In a multi-pool market, an LP can set asymmetric fees to attract flow from one direction.
For example: lower buy-side fee to attract buy flow when expecting price to rise.

**4 CP AMMs with different fee_plus / fee_minus combinations**

In [ ]:
# Symmetric baseline vs asymmetric strategies
asym_configs = [
    ('sym 30/30',   CPParams(fee_plus=0.003, fee_minus=0.003, init_x=100.0, init_y=10_000.0)),
    ('buy-heavy 10/50', CPParams(fee_plus=0.001, fee_minus=0.005, init_x=100.0, init_y=10_000.0)),
    ('sell-heavy 50/10', CPParams(fee_plus=0.005, fee_minus=0.001, init_x=100.0, init_y=10_000.0)),
    ('wide 50/50',  CPParams(fee_plus=0.005, fee_minus=0.005, init_x=100.0, init_y=10_000.0)),
]

env_asym = make_env(
    amm_specs   = [CONSTANT_PRODUCT_AMM] * 4,
    amm_params  = [cfg for _, cfg in asym_configs],
    arb_solvers = [cp_arb_solver] * 4,
    edge_fns    = [cp_edge] * 4,
    marginal_inverse_fns = [cp_inv] * 4,
)

_ = env_asym.batch_rollout(keys_warm, sim_params)
jax.block_until_ready(_)

r_asym = run_batch(env_asym)
print_results('4 CP AMMs (sym/buy-heavy/sell-heavy/wide)', r_asym)
print()
print('Individual pool configs:')
for name, cfg in asym_configs:
    print(f'  {name:<20} fee_plus={cfg.fee_plus:.3f}  fee_minus={cfg.fee_minus:.3f}')